# Jarvis — Colab T4 training
Runtime → Change runtime type → **T4 GPU**. Everything is saved to Google Drive, so if Colab disconnects just re-run all cells: training resumes from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/jarvis'
!mkdir -p {ROOT}/data {ROOT}/runs
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Get the code
Default: the code folder synced to `MyDrive/jarvis/code` (by `sync_to_drive.py` on the PC). Alternatives: set `REPO` to a GitHub repo, or put `jarvis.zip` in `MyDrive/jarvis/`.
The code is copied to local disk so Python imports are fast; checkpoints still go to Drive.

In [ ]:
REPO = ''  # optional, e.g. 'https://github.com/<you>/jarvis.git'
import os
%cd /content
!rm -rf code
if REPO:
    !git clone -q {REPO} code
elif os.path.isdir(f'{ROOT}/code'):
    !cp -r "{ROOT}/code" code
else:
    !mkdir code && unzip -q {ROOT}/jarvis.zip -d code
%cd /content/code
!pip -q install tiktoken datasets
!python -m pytest -q

## 1. World-model core (JEPA + affect + global workspace)
Small and fast — start here. Watch `survival` go up, `poison` spike after each hidden rule shift and then drop, and `S1%` rise as the fast policy learns from planning.

In [ ]:
!python train_world.py --out_dir {ROOT}/runs/world --episodes 2000 --max_minutes 170

In [ ]:
import pandas as pd
df = pd.read_json(f'{ROOT}/runs/world/log.jsonl', lines=True)
df[['survival','poison','food','S1%','mood']].rolling(20).mean().plot(subplots=True, figsize=(10,9));

In [ ]:
# H1 control run: same agent, affect has no influence. Compare logs with summarize_world.py
# !python train_world.py --out_dir {ROOT}/runs/world-noaffect --episodes 2000 --ablate_affect --max_minutes 170
!python summarize_world.py {ROOT}/runs/world/log.jsonl 100

## 2. Language backbone (Gated DeltaNet + gated attention, 3:1 hybrid)
Tokenize once (saved to Drive), then train. `tiny` ≈ 1 h sanity run; `small` is the main run. `--max_minutes` stops cleanly before Colab's limit — re-run to continue.

In [ ]:
!python prepare_data.py tinystories {ROOT}/data/tinystories --max_tokens 470000000

In [ ]:
!python train_lm.py --data_dir {ROOT}/data/tinystories --out_dir {ROOT}/runs/lm-small --preset small --batch_size 32 --block_size 512 --max_steps 20000 --max_minutes 170

### Ablations (same data, same steps) — the experiment from Notion page 15
Swap the pattern: `AAAA` (pure attention) · `RRRA` (Mamba-2-style) · `DDDA` (Gated DeltaNet hybrid) · add `--bitnet`.

In [ ]:
# !python train_lm.py --data_dir {ROOT}/data/tinystories --out_dir {ROOT}/runs/lm-attn --preset small --pattern AAAAAAAA --max_steps 5000
# !python train_lm.py --data_dir {ROOT}/data/tinystories --out_dir {ROOT}/runs/lm-bitnet --preset small --bitnet --max_steps 5000

## 3. Talk to it — affect-conditioned generation

In [ ]:
!python talk.py --lm {ROOT}/runs/lm-small/ckpt.pt --world {ROOT}/runs/world/world.pt --prompt 'Once upon a time'